#### Bibliotecas

In [2]:
import pandas as pd 

#### Arquivos da Camada Bronze

In [3]:
bronze_folder = "../bronze/"
bronze_clientes = bronze_folder + "clientes_crm.json"
bronze_custos_importacao = bronze_folder + "custos_importacao.json"
bronze_produtos_raw = bronze_folder + "produtos_raw.csv"
bronze_vendas = bronze_folder + "vendas_2023_2024.csv"

#### EDA - Tabela Clientes

In [ ]:
df_bronze_clientes = pd.read_json(bronze_clientes)

# Understanding the data and checking null values
print(df_bronze_clientes.info() , "\n")

# Checking if names are capitalized
print("Capitalized names:", "\n")
print(df_bronze_clientes["full_name"].str.istitle().value_counts(), "\n")

# Checking unique locations
print("Unique Locations:", "\n")
print(df_bronze_clientes["location"].unique(), "\n")

# Searching for duplicate codes
print("Duplicate codes", "\n" )
print(df_bronze_clientes.duplicated(subset='code').sum(), "\n")

# Checking for emails not containing '@'
print("Emails not containing @")
print(df_bronze_clientes["email"].str.contains('@').value_counts(), "\n")

#### EDA - Tabela Produtos

In [ ]:
df_bronze_produtos = pd.read_csv(bronze_produtos_raw)

# Understanding the data and checking null values
print(df_bronze_produtos.info() , "\n")
print(df_bronze_produtos.head(), "\n")

# Checking actual_category unique values
print(df_bronze_produtos["actual_category"].unique())

#### EDA - Tabela Custos 

In [ ]:
df_bronze_custos = pd.read_json(bronze_custos_importacao)

# Understanding the data and checking null values
print(df_bronze_custos.info() , "\n")
print(df_bronze_custos.head(), "\n")

# Checking unique categories
print(df_bronze_custos["category"].unique(), "\n")


#### EDA - Tabela Vendas

In [41]:
df_bronze_vendas = pd.read_csv(bronze_vendas)

# Understanding the data and checking null values
print(df_bronze_vendas.info() , "\n")
print(df_bronze_vendas.head(), "\n")


<class 'pandas.DataFrame'>
RangeIndex: 9895 entries, 0 to 9894
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id          9895 non-null   int64  
 1   id_client   9895 non-null   int64  
 2   id_product  9895 non-null   int64  
 3   qtd         9895 non-null   int64  
 4   total       9895 non-null   float64
 5   sale_date   9895 non-null   str    
dtypes: float64(1), int64(4), str(1)
memory usage: 464.0 KB
None 

   id  id_client  id_product  qtd     total   sale_date
0   0         42         105   11    3405.0  2023-09-10
1   1          3         136    9   16873.9  15-09-2024
2   2         25         139    7    9475.3  2024-08-13
3   4         20          23    5   55893.0  2023-02-03
4   5          8          57    4  451403.9  2024-02-12 



#### Tratamento de Dados

In [ ]:
treat_clientes = pd.read_json(bronze_clientes)

In [29]:
treat_produtos = pd.read_csv(bronze_produtos_raw)

treat_produtos['actual_category'] = treat_produtos['actual_category'].str.lower().str.strip()

In [ ]:
treat_vendas = pd.read_csv(bronze_vendas)

# Standardizing date columns
treat_vendas["sale_date"] = pd.to_datetime(treat_vendas["sale_date"], dayfirst=True, format="mixed", errors="coerce")

# Check for errors
print(treat_vendas["sale_date"].isna().value_counts())

sale_date
False    9895
Name: count, dtype: int64


#### Questão 1 

In [44]:
import pandas as pd
import sqlite3

bronze_folder = "../bronze/"
bronze_vendas = bronze_folder + "vendas_2023_2024.csv"

# Creates connection
conn = sqlite3.connect('../sql/vendas.db')


# 2. Uses pandas to transform csv to sql for sqlite3
df_vendas = pd.read_csv(bronze_vendas)

df_vendas.to_sql('vendas_2023_2024', conn, index=False, if_exists='replace')

# 4. Query 
query_q1 = """
SELECT 
    COUNT(*) AS total_linhas,
    (SELECT COUNT(*) FROM pragma_table_info('vendas_2023_2024')) AS total_colunas,
    MIN(sale_date) AS data_minima,
    MAX(sale_date) AS data_maxima,
    MIN(total) AS total_minimo,
    MAX(total) AS total_maximo,
    AVG(total) AS total_medio
FROM vendas_2023_2024;
"""

# 5. Display
resultado_q1 = pd.read_sql_query(query_q1, conn)

print("\nRESULTADO QUESTÃO 1.1 e 1.2\n")
print(resultado_q1, "\n")



RESULTADO QUESTÃO 1.1 e 1.2

   total_linhas  total_colunas data_minima data_maxima  total_minimo  \
0          9895              6  01-01-2023  31-12-2024         294.5   

   total_maximo    total_medio  
0     2222973.0  263797.828267   



##### Questão 1.3
> Com base na análise exploratória inicial, o dataset vendas_2023_2024.csv não é confiável em seu estado bruto e não está pronto para análises futuras:

> Inconsistência de dados: Identifiquei uma grave inconsistência de formatação na coluna temporais sale_date, que está despadronizada contendo formatos YYYY-MM-DD e DD-MM-YYYY. Como ainda não tratamos os dados, a query SQL interpreta a coluna como texto puro (String), avaliando mínimos e máximos pela ordem alfabética e distorcendo a leitura real do período.

> Possíveis outliers em "total": A coluna total apresenta uma discrepância gigantesca entre o seu valor mínimo e os seus valores máximos. Dado o contexto do catálogo da LH Nautical, que mistura itens baratos (como Cabos de Nylon) e itens de alto valor (como Motores Diesel), os limites inferior e superior estatístico para calcular outliers usando todo o dataset poderia não refletir o ticket médio para categoria de propulsão. Ainda assim, os outliers estatísticos podem posteriormente ser cruzados a partir do id de produto para garantir a veracidade da coluna "total".

#### Questão 2

In [110]:
bronze_folder = "../bronze/"
bronze_produtos_raw = bronze_folder + "produtos_raw.csv"


df_bronze_produtos = pd.read_csv(bronze_produtos_raw)

# Understanding the data and checking null values
#print(df_bronze_produtos.info() , "\n")
#print(df_bronze_produtos.head(), "\n")

# Checking actual_category unique values
#print(df_bronze_produtos["actual_category"].unique())

treat_produtos = pd.read_csv(bronze_produtos_raw)

treat_produtos['actual_category'] = treat_produtos['actual_category'].str.lower().str.strip()


def category_cleaner(cat):
    pre_clean_category = str(cat).lower().replace(' ', '')
   
    if 'ele' in pre_clean_category:
        return 'eletrônicos'
    elif 'anc' in pre_clean_category:
        return 'ancoragem'
    elif 'enc' in pre_clean_category:
        return 'ancoragem'
    elif 'pro' in pre_clean_category:
        return 'propulsão'
    else:
        return pre_clean_category

# Applying function to each field 
treat_produtos['actual_category'] = treat_produtos['actual_category'].apply(category_cleaner)

#print(f"Categorias Únicas :")
#print(treat_produtos['actual_category'].unique())

# Obs.: Valores têm '.' como decimais já que o arquivo era comma-separated csv. 
# Não me preocupei em checar valores com ',' como decimais 

def clean_prices(x):
    if pd.isna(x):
        return None
    x = str(x)
    x = x.replace('R$', '')
    x.strip()
    return float(x)

treat_produtos['price'] = treat_produtos['price'].apply(clean_prices)
#print(f"New prices :")
#print(treat_produtos['price'])

# Checking and removing duplicates 
def robust_remove_duplicates(df):
    initial_rows = len(df)

    dupes = df[df.duplicated(subset='code', keep=False)]
    
    if dupes.empty:
        print("No duplicates found.")
        return df
    
    for code in dupes['code'].unique():
        group = dupes[dupes['code'] == code]
        
        if group.drop_duplicates().shape[0] > 1:
            print(f"Inconsistent duplicates for code {code}")
            return df
        # print(f"\nDuplicatas encontradas:\n{group}")
    
    print("\nAll duplicates are identical. Safe to remove!\n")
    
    df_clean = df.drop_duplicates(subset='code', keep='first')
    removed = initial_rows - len(df_clean)
    
    print(f"Removed [{removed}] duplicate rows")
    
    return df_clean

# Apply robust_remove_duplicates and update treat_produtos with returned df_clean
treat_produtos = robust_remove_duplicates(treat_produtos)

#print(treat_produtos)
    


All duplicates are identical. Safe to remove!

Removed [7] duplicate rows


#### Questão 3

In [175]:
import pandas as pd
bronze_folder = "../bronze/"
bronze_custos_importacao = bronze_folder + "custos_importacao.json"

treat_custos = pd.read_json(bronze_custos_importacao)


# Flattening 'historic_data' repeating other fields 
treat_custos = pd.json_normalize(
    treat_custos.to_dict('records'), # Standard 'dict' failed to read the way data is structured
    record_path=['historic_data'], 
    meta=['product_id', 'product_name', 'category']
)

# Reordering columns 
cols = ['product_id', 'product_name', 'category', 'start_date', 'usd_price']
treat_custos = treat_custos[cols]

# print(treat_custos.dtypes)


# Change dtypes
treat_custos['product_id'] = treat_custos['product_id'].astype(int)

treat_custos['start_date'] = pd.to_datetime(treat_custos['start_date'], format='%d/%m/%Y')

treat_custos['product_name'] = treat_custos['product_name'].astype("string")

treat_custos['category'] = treat_custos['category'].astype("string")

# treat_custos.info()
# print(treat_custos.head(20))

print(treat_custos.info())
treat_custos.to_csv("../gold/gold_custos_importacao.csv", index=False)



<class 'pandas.DataFrame'>
RangeIndex: 1260 entries, 0 to 1259
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   product_id    1260 non-null   int64         
 1   product_name  1260 non-null   string        
 2   category      1260 non-null   string        
 3   start_date    1260 non-null   datetime64[us]
 4   usd_price     1260 non-null   float64       
dtypes: datetime64[us](1), float64(1), int64(1), string(2)
memory usage: 49.3 KB
None


#### Questão 4

In [212]:
import pandas as pd
import sqlite3

bronze_folder = "../bronze/"
silver_folder = "../silver"

bronze_vendas = bronze_folder + "vendas_2023_2024.csv"
gold_custos = pd.read_csv('../gold/gold_custos_importacao.csv')

treat_vendas = pd.read_csv(bronze_vendas)

# Standardizing date columns
treat_vendas["sale_date"] = pd.to_datetime(treat_vendas["sale_date"], dayfirst=True, format="mixed")

# Necessary to include last business day of 2022 for sales that might have happened 01/01/2023 
url_bcb_dia_30 = 'https://api.bcb.gov.br/dados/serie/bcdata.sgs.10813/dados?formato=json&dataInicial=30/12/2022&dataFinal=30/12/2022'
url_bcb = 'https://api.bcb.gov.br/dados/serie/bcdata.sgs.10813/dados?formato=json&dataInicial=01/01/2023&dataFinal=31/12/2024'

df_cambio_dia_30 = pd.read_json(url_bcb_dia_30)
df_cambio_2023_2024 = pd.read_json(url_bcb)

# Concatenate dataframes
df_cambio = pd.concat([df_cambio_dia_30, df_cambio_2023_2024], ignore_index=True)

df_cambio['data'] = pd.to_datetime(df_cambio['data'], format='%d/%m/%Y')

# Use data as index
df_cambio = df_cambio.set_index('data')


# Populate weekends with last available PTAX
calendario = pd.date_range(start='2022-12-30', end='2024-12-31')
df_cambio = df_cambio.reindex(calendario).ffill().reset_index()
df_cambio.columns = ['data', 'ptax']


df_cambio

,data,ptax
0,2022-12-30,5.2171
1,2022-12-31,5.2171
2,2023-01-01,5.2171
3,2023-01-02,5.3430
4,2023-01-03,5.3753
...,...,...
728,2024-12-27,6.1985
729,2024-12-28,6.1985
730,2024-12-29,6.1985
731,2024-12-30,6.1917
